[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kitoware/cv-project/blob/main/emotion_recognition.ipynb)

# Video-Based Emotion Recognition using CNN+LSTM on RAVDESS

This notebook implements a video-based emotion recognition system that classifies facial expressions into 8 emotional categories using temporal modeling. The pipeline:

1. **Face Detection & Alignment** — MTCNN detects and crops faces from video frames
2. **Spatial Feature Extraction** — Pretrained ResNet-18 extracts 512-d features per frame
3. **Temporal Modeling** — LSTM captures expression dynamics across 15 sampled frames
4. **Emotion Classification** — Fully connected layer outputs 8 emotion class probabilities

**Dataset:** [RAVDESS](https://zenodo.org/records/1188976) — 24 actors, 8 emotions, ~1440 speech videos

---

### Table of Contents
1. [Setup & Configuration](#1-setup--configuration)
2. [Dataset Download & Organization](#2-dataset-download--organization)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Face Detection & Preprocessing](#4-face-detection--preprocessing)
5. [Dataset & DataLoader](#5-dataset--dataloader)
6. [Model Architecture](#6-model-architecture)
7. [Training](#7-training)
8. [Evaluation](#8-evaluation)
9. [Qualitative Analysis](#9-qualitative-analysis)
10. [Discussion & Conclusion](#10-discussion--conclusion)

## 1. Setup & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q facenet-pytorch --no-deps

In [ ]:
import os
import glob
import random
import zipfile
import requests
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from facenet_pytorch import MTCNN, InceptionResnetV1

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.manifold import TSNE
from tqdm.auto import tqdm

CONFIG = {
    'num_frames': 15,
    'face_size': 224,
    'batch_size': 8,
    'num_epochs': 40,
    'learning_rate': 5e-4,
    'weight_decay': 1e-2,
    'transformer_heads': 4,
    'transformer_layers': 2,
    'transformer_dim': 256,
    'num_classes': 8,
    'dropout': 0.4,
    'grad_clip': 1.0,
    'early_stop_patience': 10,
    'mixup_alpha': 0.4,
    'num_folds': 6,
    'seed': 42,
}

BASE_DIR = Path('/content/drive/MyDrive/cv-project')
DATA_DIR = BASE_DIR / 'data' / 'ravdess'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'

for d in [DATA_DIR, PROCESSED_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
torch.cuda.manual_seed_all(CONFIG['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

EMOTION_MAP = {
    1: 'neutral',
    2: 'calm',
    3: 'happy',
    4: 'sad',
    5: 'angry',
    6: 'fearful',
    7: 'disgust',
    8: 'surprised',
}
EMOTION_NAMES = [EMOTION_MAP[i] for i in range(1, 9)]

## 2. Dataset Download & Organization

The [RAVDESS dataset](https://zenodo.org/records/1188976) contains 7,356 files from 24 professional actors (12 male, 12 female). Each filename encodes 7 attributes:

| Position | Attribute | Values |
|----------|-----------|--------|
| 1 | Modality | 01=full-AV, 02=video-only, 03=audio-only |
| 2 | Vocal channel | 01=speech, 02=song |
| 3 | Emotion | 01-08 (see mapping above) |
| 4 | Intensity | 01=normal, 02=strong |
| 5 | Statement | 01="Kids are talking...", 02="Dogs are sitting..." |
| 6 | Repetition | 01=1st, 02=2nd |
| 7 | Actor | 01-24 (odd=male, even=female) |

We download only the **speech video** files (24 actor zips).

In [ ]:
def download_ravdess(data_dir, num_actors=24):
    """Download RAVDESS speech video files from Zenodo."""
    base_url = 'https://zenodo.org/records/1188976/files'
    
    for actor_id in tqdm(range(1, num_actors + 1), desc='Downloading actors'):
        zip_name = f'Video_Speech_Actor_{actor_id:02d}.zip'
        zip_path = data_dir / zip_name
        actor_dir = data_dir / f'Actor_{actor_id:02d}'
        
        if actor_dir.exists() and any(actor_dir.glob('*.mp4')):
            continue
        
        if not zip_path.exists():
            url = f'{base_url}/{zip_name}?download=1'
            response = requests.get(url, stream=True)
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            
            with open(zip_path, 'wb') as f:
                with tqdm(total=total, unit='B', unit_scale=True, 
                          desc=zip_name, leave=False) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                        pbar.update(len(chunk))
        
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(data_dir)
        
        zip_path.unlink()

download_ravdess(DATA_DIR)

In [ ]:
def parse_ravdess_filename(filepath):
    """Parse RAVDESS filename into metadata dict."""
    name = Path(filepath).stem
    parts = name.split('-')
    return {
        'filepath': str(filepath),
        'modality': int(parts[0]),
        'vocal_channel': int(parts[1]),
        'emotion': int(parts[2]),
        'intensity': int(parts[3]),
        'statement': int(parts[4]),
        'repetition': int(parts[5]),
        'actor': int(parts[6]),
    }

all_files = sorted(glob.glob(str(DATA_DIR / 'Actor_*' / '*.mp4')))
metadata = pd.DataFrame([parse_ravdess_filename(f) for f in all_files])

# Filter: speech only (vocal_channel=1), video modalities only (1=full-AV, 2=video-only)
metadata = metadata[
    (metadata['vocal_channel'] == 1) & 
    (metadata['modality'].isin([1, 2]))
].reset_index(drop=True)

metadata['emotion_label'] = metadata['emotion'].map(EMOTION_MAP)
metadata['gender'] = metadata['actor'].apply(lambda x: 'male' if x % 2 == 1 else 'female')
# 0-indexed label for PyTorch
metadata['label'] = metadata['emotion'] - 1

print(f'Total speech video samples: {len(metadata)}')
print(f'\nEmotion distribution:')
print(metadata['emotion_label'].value_counts().sort_index())
metadata.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=metadata, x='emotion_label', hue='emotion_label', order=EMOTION_NAMES, ax=axes[0], palette='Set2', legend=False)
axes[0].set_title('Emotion Distribution')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(data=metadata, x='emotion_label', hue='gender', order=EMOTION_NAMES, ax=axes[1], palette='Set1')
axes[1].set_title('Emotion Distribution by Gender')
axes[1].set_xlabel('Emotion')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Gender')

plt.tight_layout()
plt.show()

In [ ]:
sample_video = metadata.iloc[0]['filepath']
cap = cv2.VideoCapture(sample_video)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

indices = np.linspace(0, total_frames - 1, 8, dtype=int)
frames = []
for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()

fig, axes = plt.subplots(1, len(frames), figsize=(20, 3))
sample_meta = metadata.iloc[0]
fig.suptitle(f'Sample: {sample_meta["emotion_label"]} (Actor {sample_meta["actor"]}, {sample_meta["gender"]})', fontsize=14)
for i, (ax, frame) in enumerate(zip(axes, frames)):
    ax.imshow(frame)
    ax.set_title(f'Frame {indices[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
stats = []
for _, row in tqdm(metadata.iterrows(), total=len(metadata), desc='Scanning videos'):
    cap = cv2.VideoCapture(row['filepath'])
    stats.append({
        'frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        'fps': cap.get(cv2.CAP_PROP_FPS),
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    })
    cap.release()

stats_df = pd.DataFrame(stats)
stats_df['duration'] = stats_df['frames'] / stats_df['fps']
print('Video Statistics:')
print(stats_df[['frames', 'fps', 'duration', 'width', 'height']].describe().round(2))

## 4. Face Detection & Preprocessing

For each video we:
1. Sample 15 uniformly-spaced frames
2. Detect and crop faces using MTCNN (with margin for context)
3. If a frame has no detection, fall back to the nearest detected bounding box
4. Save the result as a `.npy` array of shape `(15, 224, 224, 3)`

In [ ]:
mtcnn = MTCNN(
    image_size=CONFIG['face_size'],
    margin=40,
    min_face_size=60,
    thresholds=[0.6, 0.7, 0.7],
    post_process=False,
    device=device,
)

In [ ]:
def sample_frames(video_path, num_frames):
    """Sample num_frames evenly spaced frames from a video as PIL Images."""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    
    indices = np.linspace(0, total - 1, min(num_frames, total), dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))
    cap.release()
    
    # Pad with last frame if needed
    while len(frames) < num_frames and len(frames) > 0:
        frames.append(frames[-1])
    
    return frames[:num_frames]


def extract_faces(frames, detector, face_size=224):
    """Detect and crop faces from PIL frames with temporal fallback."""
    faces = [None] * len(frames)
    boxes = [None] * len(frames)
    
    for i, frame in enumerate(frames):
        detected_boxes, probs = detector.detect(frame)
        if detected_boxes is not None and len(detected_boxes) > 0:
            boxes[i] = detected_boxes[0]  # Take highest-confidence face
    
    detected_indices = [i for i, b in enumerate(boxes) if b is not None]
    if not detected_indices:
        return None
    
    # Fill missing detections with nearest detected box
    for i in range(len(frames)):
        if boxes[i] is None:
            nearest = min(detected_indices, key=lambda j: abs(j - i))
            boxes[i] = boxes[nearest]
    
    for i, frame in enumerate(frames):
        img = np.array(frame)
        x1, y1, x2, y2 = [int(c) for c in boxes[i]]
        # Clamp to image bounds
        h, w = img.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            # Fallback: use center crop
            cx, cy = w // 2, h // 2
            half = min(w, h) // 3
            x1, y1 = cx - half, cy - half
            x2, y2 = cx + half, cy + half
        crop = img[y1:y2, x1:x2]
        face = cv2.resize(crop, (face_size, face_size))
        faces[i] = face
    
    return np.stack(faces)  # (num_frames, face_size, face_size, 3)


def preprocess_video(video_path, detector, num_frames=15, face_size=224):
    """Full pipeline: sample frames -> detect faces -> return array."""
    frames = sample_frames(video_path, num_frames)
    if not frames:
        return None
    return extract_faces(frames, detector, face_size)

In [ ]:
failures = []

for idx, row in tqdm(metadata.iterrows(), total=len(metadata), desc='Preprocessing videos'):
    video_name = Path(row['filepath']).stem
    npy_path = PROCESSED_DIR / f'{video_name}.npy'
    
    if npy_path.exists():
        continue
    
    faces = preprocess_video(
        row['filepath'], mtcnn,
        num_frames=CONFIG['num_frames'],
        face_size=CONFIG['face_size'],
    )
    
    if faces is not None:
        np.save(npy_path, faces.astype(np.uint8))
    else:
        failures.append(row['filepath'])

print(f'\nProcessed: {len(metadata) - len(failures)} / {len(metadata)}')
print(f'Failures: {len(failures)}')
if failures:
    print('Failed files:', failures[:5])

In [ ]:
fig, axes = plt.subplots(3, CONFIG['num_frames'], figsize=(20, 5))

sample_emotions = [3, 5, 8]  # happy, angry, surprised
for row_idx, emo in enumerate(sample_emotions):
    sample = metadata[metadata['emotion'] == emo].iloc[0]
    npy_path = PROCESSED_DIR / f"{Path(sample['filepath']).stem}.npy"
    if not npy_path.exists():
        continue
    faces = np.load(npy_path)
    for col_idx in range(CONFIG['num_frames']):
        axes[row_idx, col_idx].imshow(faces[col_idx])
        axes[row_idx, col_idx].axis('off')
    axes[row_idx, 0].set_ylabel(EMOTION_MAP[emo], fontsize=12, rotation=0, labelpad=60)

plt.suptitle('Extracted Face Sequences', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Dataset & DataLoader — 6-fold actor-disjoint CV

A 2-actor test set is too noisy to evaluate small architectural changes — accuracy can swing several percent purely from split luck. We use **6-fold actor-disjoint cross-validation** instead:

- 24 actors split into 6 disjoint folds of 4 actors each
- For each fold: **4 actors test**, **4 actors val** (the next fold cyclically), **16 actors train**
- All splits are actor-disjoint, so the model never sees a test/val actor during training
- We report mean ± std test accuracy across folds, plus aggregated confusion matrix and classification report

In [ ]:
ALL_ACTORS = list(range(1, 25))  # 24 actors
NUM_FOLDS = CONFIG['num_folds']
ACTORS_PER_FOLD = len(ALL_ACTORS) // NUM_FOLDS
assert len(ALL_ACTORS) % NUM_FOLDS == 0, 'Actors must divide evenly into folds'

def get_fold_actors(fold_idx):
    """Return (train, val, test) actor IDs for a fold (actor-disjoint).
    Test = the fold_idx block; Val = the next block (cyclic); Train = the rest."""
    test_start = fold_idx * ACTORS_PER_FOLD
    val_start = ((fold_idx + 1) % NUM_FOLDS) * ACTORS_PER_FOLD
    test_actors = ALL_ACTORS[test_start:test_start + ACTORS_PER_FOLD]
    val_actors = ALL_ACTORS[val_start:val_start + ACTORS_PER_FOLD]
    held = set(test_actors) | set(val_actors)
    train_actors = [a for a in ALL_ACTORS if a not in held]
    return train_actors, val_actors, test_actors

metadata['npy_path'] = metadata['filepath'].apply(
    lambda f: str(PROCESSED_DIR / f"{Path(f).stem}.npy")
)
metadata_valid = metadata[metadata['npy_path'].apply(os.path.exists)].reset_index(drop=True)

print(f'Total metadata rows: {len(metadata)}, with existing .npy files: {len(metadata_valid)}')
if len(metadata_valid) == 0:
    raise RuntimeError(
        f"No .npy files found in {PROCESSED_DIR}. Re-run preprocessing (face detection)."
    )

print(f'\n{NUM_FOLDS}-fold actor-disjoint splits:')
for fi in range(NUM_FOLDS):
    tr, va, te = get_fold_actors(fi)
    n_tr = (metadata_valid['actor'].isin(tr)).sum()
    n_va = (metadata_valid['actor'].isin(va)).sum()
    n_te = (metadata_valid['actor'].isin(te)).sum()
    print(f'  Fold {fi}: train={len(tr)} actors / {n_tr} samples | '
          f'val={len(va)} actors / {n_va} samples | '
          f'test={len(te)} actors / {n_te} samples  '
          f'(test actors: {te})')

In [ ]:
# FaceNet (VGGFace2) preprocessing:
#  - Resize face crops to 160x160 (the input size InceptionResnetV1 was trained at)
#  - ToTensor() gives [0,1], then Normalize maps to [-1,1] to match
#    `fixed_image_standardization` used during VGGFace2 training.
FACE_INPUT_SIZE = 160
FACE_MEAN = [0.5, 0.5, 0.5]
FACE_STD = [0.5, 0.5, 0.5]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(FACE_INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=FACE_MEAN, std=FACE_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(FACE_INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=FACE_MEAN, std=FACE_STD),
])

In [ ]:
class RAVDESSDataset(Dataset):
    def __init__(self, df, transform=None, num_frames=15):
        self.samples = list(zip(df['npy_path'].values, df['label'].values))
        self.transform = transform
        self.num_frames = num_frames

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        npy_path, label = self.samples[idx]
        faces = np.load(npy_path)  # (num_frames, 224, 224, 3) uint8

        frames = []
        for i in range(self.num_frames):
            frame = faces[i]  # (224, 224, 3)
            if self.transform:
                frame = self.transform(frame)
            else:
                frame = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
            frames.append(frame)

        frames = torch.stack(frames)  # (num_frames, 3, 224, 224)
        label = torch.tensor(label, dtype=torch.long)
        return frames, label

In [ ]:
def make_loaders(fold_idx):
    """Build train/val/test DataLoaders for a given fold and return them
    along with the train DataFrame (needed for class-weight computation)."""
    train_actors, val_actors, test_actors = get_fold_actors(fold_idx)
    tr_df = metadata_valid[metadata_valid['actor'].isin(train_actors)].reset_index(drop=True)
    va_df = metadata_valid[metadata_valid['actor'].isin(val_actors)].reset_index(drop=True)
    te_df = metadata_valid[metadata_valid['actor'].isin(test_actors)].reset_index(drop=True)

    train_ds = RAVDESSDataset(tr_df, train_transform, CONFIG['num_frames'])
    val_ds = RAVDESSDataset(va_df, val_transform, CONFIG['num_frames'])
    test_ds = RAVDESSDataset(te_df, val_transform, CONFIG['num_frames'])

    return (
        DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0, pin_memory=True),
        DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=True),
        DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=True),
        tr_df,
    )

# Sanity check on fold 0
_tl, _vl, _testl, _trdf = make_loaders(0)
batch_frames, batch_labels = next(iter(_tl))
print(f'Sanity check on fold 0:')
print(f'  Train batches: {len(_tl)} | Val batches: {len(_vl)} | Test batches: {len(_testl)}')
print(f'  Batch frames shape: {batch_frames.shape}   # (B, T, 3, H, W)')
print(f'  Batch labels shape: {batch_labels.shape}')
print(f'  Frame value range: [{batch_frames.min():.2f}, {batch_frames.max():.2f}]  '
      f'(should be ~[-1, 1] after FaceNet normalization)')
del _tl, _vl, _testl, _trdf, batch_frames, batch_labels

## 6. Model Architecture

```
Input: (B, T=15, 3, 160, 160)   # 224x224 face crops resized to FaceNet's native 160x160
         │
         ▼
  FaceNet (InceptionResnetV1, VGGFace2-pretrained, frozen) ──► (B, T, 512)
         │
         ▼
  Linear Projection (512 → 256) + Positional Encoding ──► (B, T, 256)
         │
         ▼
  Transformer Encoder (2 layers, 4 heads) ──► (B, T, 256)
         │
         ▼
  Attention Pooling ──► (B, 256)  [weighted sum across timesteps]
         │
         ▼
  FC(256 → 8) ──► logits
```

**Key design choices:**
- **Face-domain backbone (FaceNet / InceptionResnetV1, VGGFace2)**: Pretrained on 3.3M face images across ~9k identities, this backbone produces features tuned to facial appearance — much more useful for FER than ImageNet features. Output dim is 512 (same as ResNet-18), so the rest of the architecture is unchanged. We keep it frozen and force its BatchNorm into eval mode whenever the wrapper is `train()`-ed (otherwise BN running stats would drift on a tiny dataset).
- **Transformer encoder**: Self-attention captures relationships between any pair of frames, enabling the model to detect temporal patterns like the onset and apex of an expression.
- **Positional encoding**: Injects frame ordering information so the Transformer knows the temporal sequence.
- **Attention pooling**: Learns to weight the most emotionally salient frames rather than treating all frames equally.
- **Mixup augmentation**: Blends pairs of training samples and labels, acting as a strong regularizer for small datasets.

In [ ]:
class CNNFeatureExtractor(nn.Module):
    """FaceNet (InceptionResnetV1, VGGFace2-pretrained), fully frozen.
    Outputs 512-d face embeddings — same dim as ResNet-18, so the projection layer
    downstream stays unchanged."""

    def __init__(self):
        super().__init__()
        self.backbone = InceptionResnetV1(pretrained='vggface2')
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.backbone.eval()

    def train(self, mode=True):
        # Keep frozen backbone in eval mode (use running BN stats) regardless of
        # whether the parent module is set to train().
        super().train(mode)
        self.backbone.eval()
        return self

    def forward(self, x):
        return self.backbone(x)  # (B, 512)


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for temporal ordering."""

    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TemporalAttentionPool(nn.Module):
    """Attention pooling over temporal dimension."""

    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x):
        # x: (B, T, hidden_size)
        scores = self.attention(x)              # (B, T, 1)
        weights = F.softmax(scores, dim=1)      # (B, T, 1)
        context = (x * weights).sum(dim=1)      # (B, hidden_size)
        return context


class EmotionRecognitionModel(nn.Module):
    """FaceNet + Transformer Encoder + Attention Pooling for video emotion recognition."""

    def __init__(self, num_classes=8, d_model=256, nhead=4, num_layers=2, dropout=0.4):
        super().__init__()
        self.cnn = CNNFeatureExtractor()

        # Project CNN features to transformer dimension
        self.projection = nn.Sequential(
            nn.Linear(512, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
        )

        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.attention_pool = TemporalAttentionPool(d_model)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        with torch.no_grad():
            features = self.cnn(x)                  # (B*T, 512)
        features = features.view(B, T, -1)          # (B, T, 512)
        features = self.projection(features)         # (B, T, 256)
        features = self.pos_encoder(features)        # (B, T, 256)
        encoded = self.transformer(features)         # (B, T, 256)
        pooled = self.attention_pool(encoded)        # (B, 256)
        out = self.dropout(pooled)
        out = self.fc(out)                           # (B, 8)
        return out

In [ ]:
def make_model():
    """Construct a fresh EmotionRecognitionModel on the target device."""
    return EmotionRecognitionModel(
        num_classes=CONFIG['num_classes'],
        d_model=CONFIG['transformer_dim'],
        nhead=CONFIG['transformer_heads'],
        num_layers=CONFIG['transformer_layers'],
        dropout=CONFIG['dropout'],
    ).to(device)

# Sanity check parameter counts
_m = make_model()
total_params = sum(p.numel() for p in _m.parameters())
trainable_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen parameters:    {total_params - trainable_params:,}')
del _m
torch.cuda.empty_cache()

In [ ]:
def make_optim(model, train_df_fold):
    """Build class-weighted CE loss + AdamW + cosine-warm-restart scheduler.
    Class weights are computed from the fold's training set."""
    train_labels = train_df_fold['label'].values
    class_weights = compute_class_weight(
        'balanced', classes=np.arange(CONFIG['num_classes']), y=train_labels
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )
    return criterion, optimizer, scheduler

print(f'Optimizer: AdamW (lr={CONFIG["learning_rate"]}, wd={CONFIG["weight_decay"]})')
print(f'Scheduler: CosineAnnealingWarmRestarts (T_0=10, T_mult=2)')
print(f'Loss:      class-weighted CE + label smoothing 0.1 (weights recomputed per fold)')

## 7. Training — 6-fold cross-validation

For each fold we build fresh DataLoaders, model, optimizer, and class weights, then train with early stopping on validation accuracy. The best checkpoint per fold is saved to `checkpoints/best_model_fold{i}.pth` and used to score the held-out test actors. We aggregate per-fold test metrics into a mean ± std summary.

> **Note:** This trains 6 full models in sequence — expect roughly 6× the runtime of a single training run. Reduce `CONFIG['num_folds']` (e.g. to 3) or `CONFIG['num_epochs']` if you need to iterate faster.

In [ ]:
def mixup_data(x, y, alpha=0.4):
    """Apply mixup augmentation: blend pairs of samples and labels."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute mixup loss as weighted combination."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, device, grad_clip=1.0, mixup_alpha=0.4):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for frames, labels in tqdm(loader, desc='Train', leave=False):
        frames, labels = frames.to(device), labels.to(device)

        # Apply mixup
        mixed_frames, labels_a, labels_b, lam = mixup_data(frames, labels, mixup_alpha)

        optimizer.zero_grad()
        outputs = model(mixed_frames)
        loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * frames.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        # For accuracy tracking, count against the dominant label
        correct += (lam * predicted.eq(labels_a).sum().item()
                    + (1 - lam) * predicted.eq(labels_b).sum().item())

    if total == 0:
        return 0.0, 0.0
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for frames, labels in tqdm(loader, desc='Eval', leave=False):
        frames, labels = frames.to(device), labels.to(device)

        outputs = model(frames)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * frames.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    if total == 0:
        raise ValueError(
            "Validation loader is empty (0 samples). "
            "Check that .npy files exist for validation actors (19-22). "
            "Re-run the preprocessing cell (face detection) and verify the files were saved."
        )
    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

In [ ]:
def train_fold(fold_idx, verbose=True):
    """Train one fold end-to-end and return its history + held-out test results."""
    if verbose:
        print(f'\n{"=" * 60}\nFOLD {fold_idx + 1}/{CONFIG["num_folds"]}\n{"=" * 60}')

    train_loader, val_loader, test_loader, train_df_fold = make_loaders(fold_idx)
    model = make_model()
    criterion, optimizer, scheduler = make_optim(model, train_df_fold)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    patience_counter = 0
    best_epoch = 0
    ckpt_path = CHECKPOINT_DIR / f'best_model_fold{fold_idx}.pth'

    for epoch in range(1, CONFIG['num_epochs'] + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, device,
            CONFIG['grad_clip'], CONFIG['mixup_alpha']
        )
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if verbose and (epoch == 1 or epoch % 5 == 0):
            lr = optimizer.param_groups[0]['lr']
            print(f'  Ep {epoch:3d} | tr_loss {train_loss:.3f} tr_acc {train_acc:.3f} | '
                  f'val_loss {val_loss:.3f} val_acc {val_acc:.3f} | lr {lr:.5f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= CONFIG['early_stop_patience']:
                if verbose:
                    print(f'  Early stop @ epoch {epoch} (best epoch {best_epoch}, val {best_val_acc:.3f})')
                break

    # Reload best checkpoint and evaluate on this fold's held-out test actors
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
    test_f1_macro = f1_score(test_labels, test_preds, average='macro')
    test_f1_weighted = f1_score(test_labels, test_preds, average='weighted')

    if verbose:
        print(f'  TEST | acc {test_acc:.4f} | f1_macro {test_f1_macro:.4f} | '
              f'f1_weighted {test_f1_weighted:.4f} | best_epoch {best_epoch}')

    del model
    torch.cuda.empty_cache()

    return {
        'fold': fold_idx,
        'history': history,
        'best_epoch': best_epoch,
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'test_loss': test_loss,
        'test_f1_macro': test_f1_macro,
        'test_f1_weighted': test_f1_weighted,
        'test_preds': test_preds,
        'test_labels': test_labels,
        'ckpt_path': str(ckpt_path),
    }


fold_results = []
for fi in range(CONFIG['num_folds']):
    fold_results.append(train_fold(fi))

print('\n' + '=' * 60)
print('K-FOLD CV SUMMARY')
print('=' * 60)
accs = np.array([r['test_acc'] for r in fold_results])
f1m = np.array([r['test_f1_macro'] for r in fold_results])
f1w = np.array([r['test_f1_weighted'] for r in fold_results])
print(f'Test acc        : {accs.mean():.4f} ± {accs.std():.4f}   '
      f'(folds: {[f"{a:.3f}" for a in accs]})')
print(f'Test F1 macro   : {f1m.mean():.4f} ± {f1m.std():.4f}')
print(f'Test F1 weighted: {f1w.mean():.4f} ± {f1w.std():.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for r in fold_results:
    ep = range(1, len(r['history']['train_loss']) + 1)
    ax1.plot(ep, r['history']['val_loss'], label=f'Fold {r["fold"]}', alpha=0.75)
    ax2.plot(ep, r['history']['val_acc'], label=f'Fold {r["fold"]}', alpha=0.75)

ax1.set_xlabel('Epoch'); ax1.set_ylabel('Validation Loss')
ax1.set_title('Validation Loss per Fold')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation Accuracy')
ax2.set_title('Validation Accuracy per Fold')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluation

We pool predictions from every fold's held-out test actors into a single set of `(label, prediction)` pairs covering all 24 actors. The classification report, confusion matrix, per-class accuracy, and gender breakdown are computed on this aggregated set, while the headline accuracy is reported as **mean ± std across folds**.

In [ ]:
all_preds = np.concatenate([r['test_preds'] for r in fold_results])
all_labels = np.concatenate([r['test_labels'] for r in fold_results])

agg_acc = (all_preds == all_labels).mean()
agg_f1_macro = f1_score(all_labels, all_preds, average='macro')
agg_f1_weighted = f1_score(all_labels, all_preds, average='weighted')

print(f'Aggregated test results across {CONFIG["num_folds"]} folds  '
      f'({len(all_labels)} samples covering all 24 actors):')
print(f'  Accuracy:       {agg_acc:.4f}')
print(f'  F1 (macro):     {agg_f1_macro:.4f}')
print(f'  F1 (weighted):  {agg_f1_weighted:.4f}')

print('\nPer-fold breakdown:')
for r in fold_results:
    print(f'  Fold {r["fold"]}: acc={r["test_acc"]:.4f}  '
          f'f1_macro={r["test_f1_macro"]:.4f}  '
          f'best_val={r["best_val_acc"]:.4f}@ep{r["best_epoch"]}')

accs = np.array([r['test_acc'] for r in fold_results])
print(f'\nMean test acc across folds: {accs.mean():.4f} ± {accs.std():.4f}')

In [ ]:
print('Classification Report (aggregated across all folds):')
print('=' * 60)
print(classification_report(all_labels, all_preds, target_names=EMOTION_NAMES, digits=3))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTION_NAMES,
            yticklabels=EMOTION_NAMES, ax=ax1)
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
ax1.set_title('Confusion Matrix (Aggregated Counts)')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=EMOTION_NAMES,
            yticklabels=EMOTION_NAMES, ax=ax2)
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
ax2.set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.show()

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(EMOTION_NAMES, per_class_acc, color=plt.cm.Set2(np.arange(8)))
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Test Accuracy (aggregated across folds)')
ax.set_ylim(0, 1.0)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=10)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Reconstruct an aggregated test DataFrame by walking each fold's test actors in order.
test_meta_parts = []
for r in fold_results:
    _, _, test_actors = get_fold_actors(r['fold'])
    fold_te = metadata_valid[metadata_valid['actor'].isin(test_actors)].reset_index(drop=True).copy()
    assert len(fold_te) == len(r['test_preds']), (
        f'Fold {r["fold"]} test_df length {len(fold_te)} != preds length {len(r["test_preds"])}'
    )
    fold_te['predicted'] = r['test_preds']
    fold_te['fold'] = r['fold']
    test_meta_parts.append(fold_te)

test_meta = pd.concat(test_meta_parts, ignore_index=True)
test_meta['correct'] = (test_meta['label'] == test_meta['predicted']).astype(int)

gender_acc = test_meta.groupby('gender')['correct'].mean()
print('Accuracy by Gender:')
for gender, acc in gender_acc.items():
    print(f'  {gender}: {acc:.4f}')

gender_emo_acc = test_meta.groupby(['gender', 'emotion_label'])['correct'].mean().unstack(fill_value=0)
gender_emo_acc.plot(kind='bar', figsize=(12, 5), rot=0)
plt.title('Accuracy by Gender and Emotion (aggregated across folds)')
plt.ylabel('Accuracy')
plt.legend(title='Emotion', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 9. Qualitative Analysis

In [ ]:
correct_mask = test_meta['correct'].values.astype(bool)
incorrect_mask = ~correct_mask


def show_predictions(indices, title):
    n = min(4, len(indices))
    if n == 0:
        print(f'No samples for: {title}')
        return
    fig, axes = plt.subplots(n, CONFIG['num_frames'], figsize=(20, 3 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(title, fontsize=14)

    for row_idx, sample_idx in enumerate(indices[:n]):
        row = test_meta.iloc[sample_idx]
        faces = np.load(row['npy_path'])
        true_label = EMOTION_NAMES[int(row['label'])]
        pred_label = EMOTION_NAMES[int(row['predicted'])]

        for col_idx in range(CONFIG['num_frames']):
            axes[row_idx, col_idx].imshow(faces[col_idx])
            axes[row_idx, col_idx].axis('off')
        axes[row_idx, 0].set_ylabel(
            f'True: {true_label}\nPred: {pred_label}\n(fold {int(row["fold"])})',
            fontsize=10, rotation=0, labelpad=80
        )

    plt.tight_layout()
    plt.show()


correct_indices = np.where(correct_mask)[0]
incorrect_indices = np.where(incorrect_mask)[0]

show_predictions(correct_indices, 'Correctly Classified Samples')
show_predictions(incorrect_indices, 'Misclassified Samples')

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    """Extract attention-pooled transformer embeddings (256-d) for all samples."""
    model.eval()
    embeddings = []
    labels = []

    for frames, batch_labels in tqdm(loader, desc='Extracting embeddings'):
        frames = frames.to(device)
        B, T, C, H, W = frames.shape
        x = frames.view(B * T, C, H, W)
        feats = model.cnn(x).view(B, T, -1)
        feats = model.projection(feats)
        feats = model.pos_encoder(feats)
        encoded = model.transformer(feats)
        pooled = model.attention_pool(encoded)  # (B, d_model)

        embeddings.append(pooled.cpu().numpy())
        labels.append(batch_labels.numpy())

    return np.concatenate(embeddings), np.concatenate(labels)


# Visualize the best-performing fold's embedding space
best_fold = max(fold_results, key=lambda r: r['test_acc'])
print(f'Running t-SNE on best fold (fold {best_fold["fold"]}, '
      f'test_acc={best_fold["test_acc"]:.4f})')

_, _, best_test_loader, _ = make_loaders(best_fold['fold'])
viz_model = make_model()
viz_model.load_state_dict(
    torch.load(best_fold['ckpt_path'], map_location=device, weights_only=True)
)

test_embeddings, test_embed_labels = extract_embeddings(viz_model, best_test_loader, device)

tsne = TSNE(
    n_components=2, random_state=CONFIG['seed'],
    perplexity=min(30, max(2, len(test_embeddings) - 1)),
)
embeddings_2d = tsne.fit_transform(test_embeddings)

plt.figure(figsize=(10, 8))
for i, emotion in enumerate(EMOTION_NAMES):
    mask = test_embed_labels == i
    if mask.any():
        plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                    label=emotion, alpha=0.7, s=50)
plt.legend(title='Emotion', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title(f't-SNE of Transformer Embeddings (Best Fold {best_fold["fold"]} Test Set)')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

del viz_model
torch.cuda.empty_cache()

## 10. Discussion & Conclusion

### Results Summary

We use a frozen **FaceNet (InceptionResnetV1, VGGFace2)** spatial backbone followed by a small Transformer encoder with attention pooling for video-based emotion recognition on RAVDESS. Because RAVDESS has only 24 actors, we evaluate with **6-fold actor-disjoint cross-validation** (4 test + 4 val + 16 train actors per fold) so that headline metrics reflect generalization to unseen identities and have a meaningful variance estimate.

**Key observations:**
- The actor-disjoint folds give a much more reliable estimate of generalization than the original single 2-actor test split, where accuracy could swing several percent purely from split luck.
- Switching from ImageNet ResNet-18 to VGGFace2-pretrained FaceNet provides spatial features tuned to facial appearance rather than generic object categories — usually a noticeable boost for FER, with no change in feature dimension or downstream architecture.
- Certain emotion pairs (e.g., calm vs. neutral, fear vs. surprise) remain commonly confused, consistent with their visual similarity.
- The temporal Transformer helps distinguish emotions that share similar peak expressions but differ in onset/offset dynamics.

### Comparison to Prior Work

State-of-the-art on RAVDESS video-only emotion recognition typically achieves 60–75% accuracy depending on methodology and evaluation protocol. Our face-pretrained backbone + Transformer pipeline provides a strong baseline within this range.

### Limitations
- **Acted emotions**: May not generalize well to spontaneous/naturalistic expressions
- **Single modality**: Uses only visual information; audio contains complementary cues
- **Actor pool**: 24 actors from one cultural background limits diversity
- **Fixed backbone resolution**: We resize to 160×160 to match FaceNet's training resolution; higher-res backbones may capture finer expression cues

### Potential Extensions
- **AffectNet-pretrained backbone**: An expression-pretrained model (EmoNet, POSTER, HSEmotion) typically beats VGGFace2 on FER tasks
- **Two-stage fine-tuning**: After the Transformer converges, unfreeze the last block of the backbone with a small LR
- **Multimodal fusion**: Combine video features with audio features (MFCCs, wav2vec) for richer representation — usually the single biggest gain available on RAVDESS
- **Auxiliary intensity head**: Multi-task learning using RAVDESS's intensity labels
- **Test-time augmentation**: Average predictions over horizontal flip + multiple frame samplings
- **Cross-dataset evaluation**: Test on FER2013, AffectNet, or CK+ to assess generalization
- **Real-time inference**: Optimize the pipeline for webcam-based real-time emotion recognition

### Applications
- Human-computer interaction and adaptive interfaces
- Mental health monitoring and therapy support
- Education technology (engagement detection)
- Entertainment and gaming

### References

1. Livingstone, S. R., & Russo, F. A. (2018). The Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS). *PLoS ONE*, 13(5), e0196391.
2. Schroff, F., Kalenichenko, D., & Philbin, J. (2015). FaceNet: A unified embedding for face recognition and clustering. *CVPR*.
3. Cao, Q., Shen, L., Xie, W., Parkhi, O. M., & Zisserman, A. (2018). VGGFace2: A dataset for recognising faces across pose and age. *FG*.
4. Vaswani, A., et al. (2017). Attention is all you need. *NeurIPS*.
5. Zhang, K., Zhang, Z., Li, Z., & Qiao, Y. (2016). Joint face detection and alignment using multitask cascaded convolutional networks. *IEEE Signal Processing Letters*, 23(10), 1499–1503.